# Лабораторная работа: декодирование DTMF сигнала
Выполнила: Перминова Анастасия, группа M4121

In [1]:
# импортируем нужные библиотеки
import numpy as np
import scipy.io.wavfile as wav
import math
import IPython.display as ipd

dtmf_low = [697.0, 770.0, 852.0, 941.0]  # низкие частоты
dtmf_high = [1209.0, 1336.0, 1477.0, 1633.0]  # высокие частоты

# словарь соответствия частот символам на клавиатуре
DTMF_KEYS = {
    (697, 1209): '1', (697, 1336): '2', (697, 1477): '3', (697, 1633): 'A',
    (770, 1209): '4', (770, 1336): '5', (770, 1477): '6', (770, 1633): 'B',
    (852, 1209): '7', (852, 1336): '8', (852, 1477): '9', (852, 1633): 'C',
    (941, 1209): '*', (941, 1336): '0', (941, 1477): '#', (941, 1633): 'D'
}

# функция реализации алгоритма Герцеля
def gerzel(samples, sample_rate, freq):

    N = len(samples)
    if N == 0:
        return 0.0

    x = samples.astype(float)

    omega = 2.0 * math.pi * freq / sample_rate  # угловая частота
    coeff = 2.0 * math.cos(omega)

    s_prev = 0.0
    s_prev2 = 0.0

    for n in range(N):
        s = x[n] + coeff * s_prev - s_prev2  # по формуле из теории
        s_prev2 = s_prev
        s_prev = s

    real = s_prev - s_prev2 * math.cos(omega)  # вычисляем действительную часть спектра
    imag = s_prev2 * math.sin(omega)  # вычисляем мнимую часть спектра

    return math.hypot(real, imag) / N


def classify_dtmf_segment(segment, fs):

    # для каждого сегмента вычисляем амплитуду всех низких и высоких DTMF частот
    lows = [(f, gerzel(segment, fs, f)) for f in dtmf_low]
    highs = [(f, gerzel(segment, fs, f)) for f in dtmf_high]

    # находим самую сильную низкую и высокую частоту в сегменте
    low_f, _ = max(lows, key=lambda p: p[1])
    high_f, _ = max(highs, key=lambda p: p[1])

    return low_f, high_f

# читаем .wav файл
orig_file = "C:/Users/Home/Desktop/DTMF_Lab8.wav"
fs, data = wav.read(orig_file)
data = data.astype(float)

# воспроизводим
player = ipd.Audio(data, rate=fs)
ipd.display(player)

tone_ms = 200  # длина тона
pause_ms = 100  # длина паузы

# переводим в количество отсчетов
tone_samples = int(tone_ms / 1000 * fs)
pause_samples = int(pause_ms / 1000 * fs)
block_size = tone_samples + pause_samples  # для перехода к следующему сегменту

decoded = []
i = 0

while i + tone_samples <= len(data):
    segment = data[i : i + tone_samples]  # выделяем сегмент

    lowf, highf = classify_dtmf_segment(segment, fs)  # определяем частоты при помощи алгоритма Герцеля
    key = DTMF_KEYS.get((lowf, highf), '?')

    decoded.append(key)

    i += block_size  # переходим к следующему сегменту

print("Декодированная последовательность:", "".join(decoded))


Декодированная последовательность: 1A2#3B4*5C6D7D8*
